# Music Recommendation Prototype using CLAP embeddings

## 1. Setup & Imports


In [ ]:
%pip install numpy torch torchaudio transformers faiss-cpu spotipy librosa soundfile --quiet


In [ ]:
import os
import tempfile
import numpy as np
import torch
import torchaudio
from transformers import AutoProcessor, ClapModel
import faiss
from spotipy import Spotify
from spotipy.oauth2 import SpotifyClientCredentials

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
model_name = "laion/clap-htsat-unfused"  # or you could pick music-specific model: laion/larger_clap_music_and_speech
processor = AutoProcessor.from_pretrained(model_name)
model = ClapModel.from_pretrained(model_name).to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/615M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/614M [00:00<?, ?B/s]

ClapModel(
  (text_model): ClapTextModel(
    (embeddings): ClapTextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ClapTextEncoder(
      (layer): ModuleList(
        (0-11): 12 x ClapTextLayer(
          (attention): ClapTextAttention(
            (self): ClapTextSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ClapTextSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

## Audio Embedding Helper

In [ ]:
import librosa
import torch
import numpy as np

# !apt-get update
# !apt-get install -y ffmpeg

def load_audio(path, sr=48000):
    # librosa reliably loads mp3/wav regardless of metadata quirks
    wav, orig_sr = librosa.load(path, sr=sr, mono=True)
    wav_tensor = torch.tensor(wav).unsqueeze(0)  # shape [1, T]
    return wav_tensor

def embed_audio(path):
    wav = load_audio(path)
    wav = wav.to(device)
    inputs = processor(audios=wav.squeeze(0).cpu().numpy(), return_tensors="pt", sampling_rate=48000).to(device)
    with torch.no_grad():
        audio_feats = model.get_audio_features(**inputs)
    return audio_feats.cpu().numpy().squeeze()



Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,125 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Pack

## Text Embedding Helper

In [ ]:
def embed_text(texts:list):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_feats = model.get_text_features(**inputs)
    return text_feats.cpu().numpy()

## Spotify API Dataset

In [ ]:
# Example: spotipy setup

spotify_id = "9fb5619d84724d66a569470cc8d03ed2"
spotify_secret = "0b21b67af7a74961858724fa729ba902"
sp = Spotify(client_credentials_manager=SpotifyClientCredentials(client_id = spotify_id, client_secret = spotify_secret))

def download_preview(sp_track_id, out_path):
    meta = sp.track(sp_track_id)
    prev_url = meta["preview_url"]
    if prev_url:
        # download the preview
        import requests
        r = requests.get(prev_url)
        open(out_path, "wb").write(r.content)
        return True
    return False

# Suppose you collect e.g. 100 track preview files named “track_{i}.mp3”

## Audio Dataset from track Online Database


In [ ]:
# # === FIXED: FREE MUSIC AUDIO SAMPLE DOWNLOAD ===
# # !pip install requests --quiet
# import os, requests, glob

# os.makedirs("./mp3dataset", exist_ok=True)

# # Ten small, open-licensed MP3s from the Free Music Archive (via Zenodo mirror)
# urls = [
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Jazz_Sampler/Kevin_MacLeod_-_01_-_Cold_Funk.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Electronic_Sampler/Kevin_MacLeod_-_01_-_Cipher.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Rock_Sampler/Kevin_MacLeod_-_01_-_Hard_Boiled.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/World_Sampler/Kevin_MacLeod_-_01_-_Agogo.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Blues_Sampler/Kevin_MacLeod_-_01_-_Backbay_Lounge.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Pop_Sampler/Kevin_MacLeod_-_01_-_Carefree.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Classical_Sampler/Kevin_MacLeod_-_01_-_Canon_in_D_Major.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Cinematic_Sampler/Kevin_MacLeod_-_01_-_Impact_Moderato.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Country_Sampler/Kevin_MacLeod_-_01_-_Crossing_the_Creek.mp3",
#     "https://files.freemusicarchive.org/storage-freemusicarchive-org/music/no_curator/Kevin_MacLeod/Folk_Sampler/Kevin_MacLeod_-_01_-_Fiddle_De_De.mp3",
# ]

# for url in urls:
#     filename = os.path.basename(url)
#     out_path = os.path.join("./mp3dataset", filename)
#     if not os.path.exists(out_path):
#         r = requests.get(url)
#         open(out_path, "wb").write(r.content)
#         print(f"✅ Downloaded {filename}")

# # Build list_of_paths
# list_of_paths = glob.glob("./mp3dataset/*.mp3")
# print(f"\nTotal tracks: {len(list_of_paths)}")
# print("Example paths:\n", list_of_paths[:3])



Total tracks: 10
Example paths:
 ['/content/audio/Kevin_MacLeod_-_01_-_Cold_Funk.mp3', '/content/audio/Kevin_MacLeod_-_01_-_Backbay_Lounge.mp3', '/content/audio/Kevin_MacLeod_-_01_-_Carefree.mp3']


##Database from local MP3 files

In [ ]:
# open local zip folder
# from google.colab import files
# uploaded = files.upload()  # choose your mp3dataset.zip


Saving mp3dataset.zip to mp3dataset.zip


In [ ]:
# unzip and store
import os, shutil, glob, zipfile

audio_folder = "./mp3dataset"

# # Remove all files and directories inside the folder
# for f in os.listdir(audio_folder):
#     path = os.path.join(audio_folder, f)
#     if os.path.isfile(path):
#         os.remove(path)
#     elif os.path.isdir(path):
#         shutil.rmtree(path)  # recursively delete directories

print(f"✅ Cleared all files and subdirectories from {audio_folder}")

# zip_path = "/content/mp3dataset.zip"  # path to uploaded zip
# extract_path = "/content/audio"     # folder to extract to
os.makedirs(extract_path, exist_ok=True)

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

print(f"✅ Extracted files to {extract_path}")


✅ Cleared all files and subdirectories from /content/audio
✅ Extracted files to /content/audio


In [ ]:
#build list of file paths
import glob, os

audio_folder = "./mp3dataset"

# Recursively find all MP3s in subfolders
list_of_paths = glob.glob(os.path.join(audio_folder, "**", "*.mp3"), recursive=True)

print(f"Total tracks: {len(list_of_paths)}")
print("Example paths:\n", list_of_paths[:5])



Total tracks: 28
Example paths:
 ['/content/audio/mp3dataset/MarioCart - Game Over.mp3', '/content/audio/mp3dataset/Plain.mp3', '/content/audio/mp3dataset/01 te.mp3', '/content/audio/mp3dataset/Microdosing via stannova.com.mp3', '/content/audio/mp3dataset/Por si te llega_6.mp3']


##Database Checker

In [ ]:
import os

for path in list_of_paths:
    if not os.path.isfile(path):
        print(f"❌ File not found: {path}")
    else:
        print(f" File found: {path}")


wav, sr = librosa.load("./mp3dataset/Plain.mp3", sr=16000)
print(wav.shape, sr)


 File found: /content/audio/mp3dataset/MarioCart - Game Over.mp3
 File found: /content/audio/mp3dataset/Plain.mp3
 File found: /content/audio/mp3dataset/01 te.mp3
 File found: /content/audio/mp3dataset/Microdosing via stannova.com.mp3
 File found: /content/audio/mp3dataset/Por si te llega_6.mp3
 File found: /content/audio/mp3dataset/LANDR-Die with the stars-High-Balanced.mp3
 File found: /content/audio/mp3dataset/blur test_2.mp3
 File found: /content/audio/mp3dataset/dwabtme test.mp3
 File found: /content/audio/mp3dataset/Thunder 1.mp3
 File found: /content/audio/mp3dataset/LANDR-Locked Down new mix_5-High-Balanced.mp3
 File found: /content/audio/mp3dataset/αªàαª▓αºçαºùαªòαª┐αªò αª¼αª╛αª╕αºìαªñαª¼.mp3
 File found: /content/audio/mp3dataset/Future Bliss.mp3
 File found: /content/audio/mp3dataset/TravisScottSTOPTRYINGTOBEGODAudio-arabsongtop.mp3
 File found: /content/audio/mp3dataset/Consciousness Rabbit - pATCHES.mp3
 File found: /content/audio/mp3dataset/Instrumental - Renewed Horizons

## FAISS Embedding

In [ ]:
# Assume you have n tracks, each saved as files list_of_paths
embeddings = []
ids = []
for idx, path in enumerate(list_of_paths):
    vec = embed_audio(path)
    embeddings.append(vec)
    ids.append(os.path.basename(path))
    print(f"✅ Embedded {idx+1}/{len(list_of_paths)}: {os.path.basename(path)}")

embeddings = np.vstack(embeddings).astype('float32')



/tmp/ipython-input-817856500.py:17: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = processor(audios=wav.squeeze(0).cpu().numpy(), return_tensors="pt", sampling_rate=48000).to(device)


✅ Embedded 1/28: MarioCart - Game Over.mp3
✅ Embedded 2/28: Plain.mp3
✅ Embedded 3/28: 01 te.mp3
✅ Embedded 4/28: Microdosing via stannova.com.mp3
✅ Embedded 5/28: Por si te llega_6.mp3
✅ Embedded 6/28: LANDR-Die with the stars-High-Balanced.mp3
✅ Embedded 7/28: blur test_2.mp3
✅ Embedded 8/28: dwabtme test.mp3
✅ Embedded 9/28: Thunder 1.mp3
✅ Embedded 10/28: LANDR-Locked Down new mix_5-High-Balanced.mp3
✅ Embedded 11/28: αªàαª▓αºçαºùαªòαª┐αªò αª¼αª╛αª╕αºìαªñαª¼.mp3
✅ Embedded 12/28: Future Bliss.mp3
✅ Embedded 13/28: TravisScottSTOPTRYINGTOBEGODAudio-arabsongtop.mp3
✅ Embedded 14/28: Consciousness Rabbit - pATCHES.mp3
✅ Embedded 15/28: Instrumental - Renewed Horizons ext v2 - 131.5bpm - Dmaj.mp3
✅ Embedded 16/28: Cool Song 7_14.mp3
✅ Embedded 17/28: Chase the Light.mp3
✅ Embedded 18/28: Terminal B via stannova.com.mp3
✅ Embedded 19/28: anishghosh_morningtide.mp3
✅ Embedded 20/28: TLOL_3.mp3
✅ Embedded 21/28: 070 shake accusations -8711321879629747536.mp3
✅ Embedded 22/28: Sun Machine 

In [ ]:
import faiss
import numpy as np

embeddings = np.vstack(embeddings).astype('float32')
faiss.normalize_L2(embeddings)  # normalize if you plan to use cosine similarity

d = embeddings.shape[1]          # embedding dimension
index = faiss.IndexFlatIP(d)     # inner product = cosine similarity if normalized
index.add(embeddings)            # add all embeddings to the index
print(f"✅ Added {index.ntotal} embeddings to the index")


✅ Added 28 embeddings to the index


## Query Recommendation

In [ ]:
# Example: embed a text query
query_text = ["chill hip hop song with fat 808s"]
qvec = embed_text(query_text)[0].astype('float32')
faiss.normalize_L2(qvec.reshape(1, -1))

k = 5
distances, indices = index.search(qvec.reshape(1, -1), k)
print("Top-k indices:", indices)
print("Tracks:", [ids[i] for i in indices[0]])
print("Scores:", distances)

Top-k indices: [[ 4 11 19 17 14]]
Tracks: ['Por si te llega_6.mp3', 'Future Bliss.mp3', 'TLOL_3.mp3', 'Terminal B via stannova.com.mp3', 'Instrumental - Renewed Horizons ext v2 - 131.5bpm - Dmaj.mp3']
Scores: [[0.5652543  0.55745304 0.52999055 0.5182977  0.5044085 ]]


## Wrap into simple function

In [ ]:
def recommend_by_audio(path, k=5):
    q = embed_audio(path).astype('float32')
    faiss.normalize_L2(q.reshape(1,-1))
    d, idx = index.search(q.reshape(1,-1), k)
    return [(ids[i], float(d[0][j])) for j,i in enumerate(idx[0])]

def recommend_by_text(prompt, k=5):
    q = embed_text([prompt])[0].astype('float32')
    faiss.normalize_L2(q.reshape(1,-1))
    d, idx = index.search(q.reshape(1,-1), k)
    return [(ids[i], float(d[0][j])) for j,i in enumerate(idx[0])]

##Quick Test

In [ ]:
print("Recommendations for prompt:", recommend_by_text("female vocals"))
#print("Recommendations for a seed audio:", recommend_by_audio("some_path/to/seed.wav"))

NameError: name 'recommend_by_text' is not defined

##Save Index & Metadata

In [ ]:
faiss.write_index(index, "clap_music_index.faiss")
import pickle
with open("ids.pkl", "wb") as f:
    pickle.dump(ids, f)